In [ ]:
!git clone https://github.com/JhosC-137/Sistema-Peaje-Inteligente.git
%cd Sistema-Peaje-Inteligente/inferencia
!pip install -q fastapi uvicorn python-multipart ultralytics fast-plate-ocr onnxruntime

Cloning into 'Sistema-Peaje-Inteligente'...
remote: Enumerating objects: 464, done.
remote: Counting objects: 100% (464/464), done.
remote: Compressing objects: 100% (302/302), done.
remote: Total 464 (delta 117), reused 446 (delta 102), pack-reused 0 (from 0)
Receiving objects: 100% (464/464), 49.66 MiB | 18.16 MiB/s, done.
Resolving deltas: 100% (117/117), done.
/content/Sistema-Peaje-Inteligente/inferencia/Sistema-Peaje-Inteligente/inferencia


In [ ]:
!nohup uvicorn main:app --host 0.0.0.0 --port 8000 > uvicorn.log 2>&1 &
!sleep 5
!curl -s http://localhost:8000/health

{"status":"ok","device":"cpu","modelos_cargados":{"m1_detector_onnx":true,"m1_ocr_onnx":true,"m2_vehiculos":true,"m3_ejes":true}}

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8000

2026-11-24T20:00:05Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-24T20:00:05Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-24T20:00:08Z INF +--------------------------------------------------------------------------------------------+
2026-11-24T20:00:08Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-24T20:00:08Z INF |  https://convertible-mouth-campaign-tools.trycloudflar

In [ ]:
import subprocess, time, urllib.request
from pathlib import Path

rutas = list(Path("/content").rglob("inferencia/main.py"))
assert rutas, "No se encontró inferencia/main.py. Vuelve a ejecutar el git clone."
WORK_DIR = rutas[0].parent

subprocess.run(["pkill", "-f", "uvicorn"], check=False)
log_file = open(WORK_DIR / "uvicorn.log", "w")
subprocess.Popen(
    ["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=str(WORK_DIR),
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

for _ in range(15):
    try:
        res = urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2).read().decode()
        print("Servidor activo:", res)
        break
    except Exception:
        time.sleep(1)
else:
    print("Error al arrancar uvicorn. Revisa el log:")
    print((WORK_DIR / "uvicorn.log").read_text())

KeyboardInterrupt: 

In [ ]:
!tail -n 25 /content/Sistema-Peaje-Inteligente/inferencia/uvicorn.log

INFO:     Started server process [10336]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:56906 - "GET /health HTTP/1.1" 200 OK
INFO:     74.220.48.219:0 - "POST /inferir HTTP/1.1" 422 Unprocessable Content
INFO:     74.220.48.219:0 - "POST /inferir HTTP/1.1" 422 Unprocessable Content
INFO:     2800:cd0:1e10:c700:6408:f212:26f9:27c2:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2800:cd0:1e10:c700:6408:f212:26f9:27c2:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     2800:cd0:1e10:c700:6408:f212:26f9:27c2:0 - "GET /health HTTP/1.1" 200 OK
INFO:     74.220.48.219:0 - "POST /inferir HTTP/1.1" 422 Unprocessable Content
INFO:     74.220.48.219:0 - "POST /inferir HTTP/1.1" 200 OK
INFO:     Shutting down


In [ ]:
import subprocess, time, re, urllib.request
from pathlib import Path

# 1. Actualizar repo (o clonar si el entorno de Colab se reinició)
repo_dir = Path("/content/Sistema-Peaje-Inteligente")
if repo_dir.exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/JhosC137/Sistema-Peaje-Inteligente.git", str(repo_dir)], check=True)

WORK_DIR = repo_dir / "inferencia"
!pip install -q fastapi uvicorn python-multipart ultralytics fast-plate-ocr onnxruntime websockets

# 2. Reiniciar FastAPI en IPv4
subprocess.run(["pkill", "-f", "uvicorn"], check=False)
time.sleep(1)
log_uv = open(WORK_DIR / "uvicorn.log", "w")
subprocess.Popen(
    ["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=str(WORK_DIR),
    stdout=log_uv,
    stderr=subprocess.STDOUT,
)

for _ in range(20):
    try:
        res = urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2).read().decode()
        print("FastAPI activo:", res)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Error en uvicorn:\n" + (WORK_DIR / "uvicorn.log").read_text())

# 3. Abrir túnel Cloudflare en segundo plano
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
log_cf = open("/content/cf.log", "w")
subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=log_cf,
    stderr=subprocess.STDOUT,
)

for _ in range(15):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", Path("/content/cf.log").read_text())
    if m:
        print("URL pública lista:", m.group(0))
        break

FastAPI activo: {"status":"ok","device":"cpu","modelos_cargados":{"m1_detector_onnx":true,"m1_ocr_onnx":true,"m2_vehiculos":true,"m3_ejes":true}}
URL pública lista: https://brian-staff-thinkpad-animals.trycloudflare.com


In [ ]:
!cat /content/Sistema-Peaje-Inteligente/inferencia/uvicorn.log

INFO:     Started server process [10336]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:56906 - "GET /health HTTP/1.1" 200 OK
INFO:     74.220.48.219:0 - "POST /inferir HTTP/1.1" 422 Unprocessable Content
INFO:     74.220.48.219:0 - "POST /inferir HTTP/1.1" 422 Unprocessable Content
INFO:     2800:cd0:1e10:c700:6408:f212:26f9:27c2:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2800:cd0:1e10:c700:6408:f212:26f9:27c2:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     2800:cd0:1e10:c700:6408:f212:26f9:27c2:0 - "GET /health HTTP/1.1" 200 OK
INFO:     74.220.48.219:0 - "POST /inferir HTTP/1.1" 422 Unprocessable Content
INFO:     74.220.48.219:0 - "POST /inferir HTTP/1.1" 200 OK
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [10336]
